# P14 NB01 — Sensor Anomaly Detection with Autoencoders

**Problem:** Detect industrial sensor anomalies using *unsupervised* deep learning — no labeled fault data required.

**What We Build:**
1. Vanilla Autoencoder from scratch (NumPy)
2. Variational Autoencoder (VAE) in PyTorch
3. Reparameterization trick & ELBO derivation
4. Anomaly scoring via reconstruction error
5. Latent space interpolation & visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
ROOT = Path('..').resolve()
ASSETS = ROOT / 'assets'
ASSETS.mkdir(exist_ok=True)

# Load sensor anomaly data
data = np.load(ROOT / 'data' / 'sensor_anomaly_images.npz')
images, labels = data['images'], data['labels']
print(f'Dataset: {images.shape[0]} images, {images.shape[1]}x{images.shape[2]}')
print(f'Normal: {(labels==0).sum()}, Anomaly: {(labels==1).sum()}')

In [ ]:
# ── EDA: Sample Images ──
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sensor Images — Normal (top) vs Anomaly (bottom)', fontsize=14, fontweight='bold')
normals = images[labels == 0][:8]
anomalies = images[labels == 1][:8]
for i in range(8):
    axes[0, i].imshow(normals[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Normal {i+1}', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(anomalies[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Anomaly {i+1}', fontsize=8)
    axes[1, i].axis('off')
plt.tight_layout()
plt.savefig(ASSETS / 'proj1_sensor_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════
# VANILLA AUTOENCODER FROM SCRATCH (NumPy)
# ══════════════════════════════════════════════
# A minimal autoencoder to understand the concept before going deep.
# We flatten 64x64 images to 4096-d and train a 4096 -> 128 -> 4096 bottleneck.

class VanillaAutoencoder:
    def __init__(self, input_dim=4096, hidden_dim=128, lr=0.001):
        self.lr = lr
        scale = np.sqrt(2 / input_dim)
        self.W1 = np.random.randn(input_dim, hidden_dim).astype(np.float32) * scale
        self.b1 = np.zeros(hidden_dim, dtype=np.float32)
        self.W2 = np.random.randn(hidden_dim, input_dim).astype(np.float32) * scale
        self.b2 = np.zeros(input_dim, dtype=np.float32)

    def encode(self, X):
        self._h_pre = X @ self.W1 + self.b1
        self._h = np.maximum(0, self._h_pre)  # ReLU
        return self._h

    def decode(self, h):
        return 1 / (1 + np.exp(-(h @ self.W2 + self.b2)))  # Sigmoid

    def forward(self, X):
        h = self.encode(X)
        return self.decode(h)

    def train_step(self, X):
        recon = self.forward(X)
        loss = np.mean((recon - X) ** 2)
        # Backprop
        d_out = 2 * (recon - X) / X.shape[0]
        d_out *= recon * (1 - recon)  # sigmoid derivative
        dW2 = self._h.T @ d_out
        db2 = d_out.sum(axis=0)
        d_h = d_out @ self.W2.T
        d_h[self._h_pre <= 0] = 0  # ReLU derivative
        dW1 = X.T @ d_h
        db1 = d_h.sum(axis=0)
        # Update
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        return loss

# Train on normal images only
X_flat = images[labels == 0].reshape(-1, 64*64)
ae = VanillaAutoencoder(4096, 128, lr=0.0005)
losses = []
for epoch in range(50):
    perm = np.random.permutation(len(X_flat))
    epoch_loss = 0
    for i in range(0, len(X_flat), 128):
        batch = X_flat[perm[i:i+128]]
        epoch_loss += ae.train_step(batch)
    losses.append(epoch_loss / (len(X_flat) // 128))
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/50 — MSE: {losses[-1]:.6f}')

# Show reconstructions
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
fig.suptitle('Vanilla AE — Original (top) vs Reconstruction (bottom)', fontsize=13, fontweight='bold')
sample = X_flat[:6]
recon = ae.forward(sample)
for i in range(6):
    axes[0, i].imshow(sample[i].reshape(64, 64), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].reshape(64, 64), cmap='gray')
    axes[1, i].axis('off')
plt.tight_layout()
plt.savefig(ASSETS / 'proj1_sensor_vanilla_ae_recon.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════
# VARIATIONAL AUTOENCODER (PyTorch)
# ══════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
LATENT_DIM = 32
print(f'Device: {DEVICE}')

class Encoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128*8*8, latent_dim)
        self.fc_logvar = nn.Linear(128*8*8, latent_dim)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128*8*8)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1), nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z).view(-1, 128, 8, 8)
        return self.deconv(h)

class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        # The reparameterization trick:
        # z = mu + sigma * epsilon, where epsilon ~ N(0,1)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    # ELBO = E[log p(x|z)] - KL(q(z|x) || p(z))
    recon = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl

print('VAE architecture defined.')

In [ ]:
# ── Train VAE ──
X_train = images[labels == 0].astype(np.float32)
X_train = torch.tensor(X_train[:, np.newaxis, :, :]).to(DEVICE)

model = VAE(LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 40
BATCH_SIZE = 64
train_losses = []

model.train()
for epoch in range(EPOCHS):
    perm = torch.randperm(len(X_train))
    total_loss = 0
    for i in range(0, len(X_train), BATCH_SIZE):
        batch = X_train[perm[i:i+BATCH_SIZE]]
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar) / len(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    n_batches = (len(X_train) + BATCH_SIZE - 1) // BATCH_SIZE
    avg_loss = total_loss / n_batches
    train_losses.append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.2f}')

# Save model
torch.save(model.state_dict(), ROOT / 'models' / 'vae_model.pt')
print('VAE training complete.')

In [ ]:
# ── Training Curve + VAE Reconstructions ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
ax1.plot(train_losses, color='#1E40AF', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('ELBO Loss')
ax1.set_title('VAE Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Reconstructions
model.eval()
with torch.no_grad():
    sample = X_train[:8]
    recon, _, _ = model(sample)
    sample_np = sample.cpu().squeeze(1).numpy()
    recon_np = recon.cpu().squeeze(1).numpy()

combined = np.concatenate([sample_np[:4], recon_np[:4]], axis=0)
grid_img = np.block([[combined[i] for i in range(4)],
                      [combined[i+4] for i in range(4)]])
ax2.imshow(grid_img, cmap='gray')
ax2.set_title('VAE: Original (top) vs Reconstruction (bottom)', fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig(ASSETS / 'proj1_sensor_vae_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════
# ANOMALY DETECTION via Reconstruction Error
# ══════════════════════════════════════════════
model.eval()
all_images = torch.tensor(images[:, np.newaxis, :, :].astype(np.float32)).to(DEVICE)

with torch.no_grad():
    recon_all, _, _ = model(all_images)
    recon_errors = ((recon_all - all_images) ** 2).mean(dim=(1, 2, 3)).cpu().numpy()

# Score distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of reconstruction errors
normal_errors = recon_errors[labels == 0]
anomaly_errors = recon_errors[labels == 1]
ax1.hist(normal_errors, bins=50, alpha=0.7, label='Normal', color='#22C55E', density=True)
ax1.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly', color='#EF4444', density=True)
threshold = np.percentile(normal_errors, 95)
ax1.axvline(threshold, color='k', linestyle='--', linewidth=2, label=f'Threshold={threshold:.4f}')
ax1.set_xlabel('Reconstruction Error (MSE)')
ax1.set_ylabel('Density')
ax1.set_title('Anomaly Score Distribution', fontweight='bold')
ax1.legend()

# ROC-style scatter
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
auc_score = roc_auc_score(labels, recon_errors)
precision, recall, _ = precision_recall_curve(labels, recon_errors)
pr_auc = auc(recall, precision)

ax2.scatter(range(len(recon_errors)), recon_errors, c=labels, cmap='RdYlGn_r', alpha=0.4, s=5)
ax2.axhline(threshold, color='k', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('Reconstruction Error')
ax2.set_title(f'Anomaly Scores — ROC-AUC: {auc_score:.3f}, PR-AUC: {pr_auc:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig(ASSETS / 'proj1_sensor_anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC-AUC: {auc_score:.4f}')
print(f'PR-AUC: {pr_auc:.4f}')

In [ ]:
# ══════════════════════════════════════════════
# LATENT SPACE VISUALIZATION
# ══════════════════════════════════════════════
from sklearn.manifold import TSNE

model.eval()
with torch.no_grad():
    mu, _ = model.encoder(all_images)
    z_all = mu.cpu().numpy()

# t-SNE of latent space
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
z_2d = tsne.fit_transform(z_all[:2000])  # subsample for speed
labels_sub = labels[:2000]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

scatter = ax1.scatter(z_2d[:, 0], z_2d[:, 1], c=labels_sub, cmap='RdYlGn_r', alpha=0.6, s=8)
ax1.set_title('t-SNE of VAE Latent Space', fontweight='bold')
ax1.set_xlabel('t-SNE 1')
ax1.set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=ax1, label='0=Normal, 1=Anomaly')

# Latent space interpolation
z1 = torch.randn(1, LATENT_DIM, device=DEVICE)
z2 = torch.randn(1, LATENT_DIM, device=DEVICE)
n_interp = 10
interp_images = []
with torch.no_grad():
    for alpha in np.linspace(0, 1, n_interp):
        z = (1 - alpha) * z1 + alpha * z2
        img = model.decoder(z).cpu().squeeze().numpy()
        interp_images.append(img)

interp_grid = np.concatenate(interp_images, axis=1)
ax2.imshow(interp_grid, cmap='gray')
ax2.set_title('Latent Space Interpolation (z₁ → z₂)', fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig(ASSETS / 'proj1_sensor_latent_space.png', dpi=150, bbox_inches='tight')
plt.show()

# Save interpolation as GIF
import imageio
frames = [(img * 255).astype(np.uint8) for img in interp_images]
imageio.mimsave(ASSETS / 'proj1_sensor_proj1_sensor_latent_interpolation.gif', frames, duration=0.3, loop=0)
print('Saved proj1_sensor_proj1_sensor_latent_interpolation.gif')